# **Task 1: Word2Vec with Negative Sampling**

## 1. Concept

## **Objective**
In this assignment, you will implement the **Word2Vec model from scratch** using the **negative sampling technique**.  
By the end of this part, you will:
- Understand the concept of distributed word embeddings.
- Learn about **Skip-gram** and **CBOW** architectures.
- Implement forward pass loss computation, gradient computation, backpropagation, SGD optimization and minibatch-SGD manually.
- Train Word2Vec on a text corpus with negative sampling.
- Save trained embeddings into pickle files.
- Evaluate embeddings on an **analogy task** using a provided dataset.

---

## **1. Introduction to Word2Vec**

### **What is Word2Vec?**
Word2Vec is a shallow neural network that learns **dense vector representations of words** such that semantically similar words lie close together in the embedding space.

The key idea: instead of representing words as one-hot vectors, we represent them as low-dimensional **embeddings** learned by predicting context words from a target word (or vice versa).

---
### Skip-gram Architecture

The **Skip-gram model** in Word2Vec learns word embeddings by predicting context words given a center (target) word. Instead of using one-hot vectors, we use Label Encoding, which is more memory-efficient and computationally optimal.

---

#### Visual Overview

**1. Skip-gram Model Structure**  
![word2vec Model architecture](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*tmyks7pjdwxODh5-gL3FHQ.png)
<!-- ![Skip-gram Model Structure](https://media.geeksforgeeks.org/wp-content/uploads/Skip-gram-architecture-2.jpg) -->

**2. Training Objective Illustration**  
![Skip-gram Objective](https://media.geeksforgeeks.org/wp-content/uploads/word2vec_diagram-1.jpg)

---

####  Intuition

Sentence:  
**"The cat sat on the mat"**, with **window size = 2**

If target word = `"cat"`, context = `["The", "sat"]`  
→ Training pairs:
- (`cat` → `The`)
- (`cat` → `sat`)

#### Architecture and Training Flow (with Index-based Input)

Let:  
- \( V \): size of the vocabulary  
- \( d \): embedding dimension  
-  t in $ 0, 1, \dots, V-1 $: index of the target word  
-  $E \in {R}^{V \times d}$: input embedding matrix  
-  $E' \in {R}^{V \times d}$: output embedding matrix

The integer index \( t \) is used to perform a direct embedding lookup in matrix \( E \), avoiding one-hot vector multiplication.





---

####  Step-by-Step Computation:

1. **Input**: Integer ID of target word \( t \)

2. **Embedding Lookup** (instead of one-hot):

$$
u = E[t] \in \mathbb{R}^d
$$

3. **Score for Context Word \( c \)**:

$$
\text{score}(c) = u^\top E'[c]
$$

4. **Softmax over Vocabulary**:

$$
P(w_c \mid w_t) = \frac{\exp(u^\top E'[c])}{\sum_{w=1}^{V} \exp(u^\top E'[w])}
$$

---



#### Objective Function

Given a **center word** $ w_t $ , the Skip-gram model aims to **predict surrounding context words** within a fixed window size.


$$
{L}_{\text{skip-gram}} = \sum_{-k \le j \le k,\ j \ne 0} \log P(w_{t+j} \mid w_t)
$$


where


| Term &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;| Description |
|------|-------------|
| $ {L}_{{skip-gram}} $ | The **log-likelihood** objective for predicting the context words surrounding the center word $ w_t $ This is the quantity we aim to **maximize** during training. |
| $ k $| The **window size**, which defines how many words before and after the center word are considered as context. |
| $ j $ | The **offset** from the center word $ t $, ranging from $ -k $ to $ +k $, excluding $ j = 0 $ (which would be the center word itself). |
| $ w_t $ | The **center word** (target word) whose surrounding words we are trying to predict. |
| $ w_{t+j} $ | A **context word** at position $ t+j $ relative to the center word $ w_t $ |
| $ P(w_{t+j} \mid w_t) $ | The **conditional probability** that word $ w_{t+j} $ appears in the context of $ w_t $. This is typically modeled using a softmax function or approximated using negative sampling. |
| $ \log P(w_{t+j} \mid w_t) $ | The **log-probability** of correctly predicting a context word. Using the log helps with numerical stability and converts the product of probabilities into a sum. |



---




#### Parameters Learned

- $ E \in {R}^{V \times d} $: Embedding matrix (input)
- $ E' \in {R}^{V \times d} $: Output matrix (context prediction)

**Embedding Lookup** via integer index replaces costly one-hot vector multiplication.

**Note**: Final embeddings are typically taken from matrix $ E $
---


#### Summary

- **Input**: Integer index of target word  
- **Output**: Predict context word indices  
- **Lookup**: Directly fetch embedding vector from matrix  
- **Train**: Using softmax or negative sampling  
- **Goal**: Words appearing in similar contexts should have similar vectors

---



### CBOW Architecture

The **CBOW model** in Word2Vec learns word embeddings by predicting a center (target) word given its surrounding context words. Like Skip-gram, it avoids one-hot vectors using index-based embedding lookup, making it both space- and time-efficient.

---

#### Visual Overview

**1. CBOW Model Structure**  
![CBOW Model Structure](https://media.licdn.com/dms/image/v2/D4D12AQG9BzheEOLwpg/article-cover_image-shrink_720_1280/article-cover_image-shrink_720_1280/0/1710832109081?e=1760572800&v=beta&t=3W-eSSjvNqSbz-AKtefhhbQXDFizyohSjyAWaZLek1I)



---

#### Intuition

Sentence:  
**"The cat sat on the mat"**, with **window size = 2**

If target word = `"cat"`, context = `["The", "sat"]`  
→ Training pairs:
- (`["The", "sat"]` → `cat`)

#### Architecture and Training Flow (with Index-based Input)

Let:  
- $ V $: size of the vocabulary  
- $ d $: embedding dimension  
-   C = $ c_1, c_2, \dots, c_m $ : list of context word indices  
-  $ t $: index of the target word  
-  $ E \in {R}^{V \times d} $ : input embedding matrix  
-  $ E' \in {R}^{V \times d} $ : output embedding matrix

Each context word $ c_i \in C $ is used to lookup a vector from $ E $, which are then averaged to form a single context embedding.

---

#### Step-by-Step Computation:

1. **Input**: Integer IDs of context words  C = $ {c_1, c_2, \dots, c_m} $

2. **Embedding Lookup**:

$$
h = \frac{1}{m} \sum_{i=1}^{m} E[c_i] \in \mathbb{R}^d
$$

3. **Score for Target Word $ t $**:

$$
\text{score}(t) = h^\top E'[t]
$$

4. **Softmax over Vocabulary**:

$$
P(w_t \mid \text{context}) = \frac{\exp(h^\top E'[t])}{\sum_{w=1}^{V} \exp(h^\top E'[w])}
$$

---

#### Objective Function

Given **context words** $ \{w_{t-k}, \dots, w_{t-1}, w_{t+1}, \dots, w_{t+k}\} $, the CBOW model aims to **predict the center word** \( w_t \).

$$
{L}_{\text{CBOW}} = \log P(w_t \mid w_{t-k}, \dots, w_{t-1}, w_{t+1}, \dots, w_{t+k})
$$

where


| Term &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;    | Description |
|---------|-------------|
| $ {L}_{\text{CBOW}} $ | The **log-likelihood** objective for predicting the center word $ w_t $ given its surrounding context words. |
| $ k $ | The **window size**, which defines how many words before and after the target word are used as context. |
| $ c_i $ | The **context word index** used for embedding lookup. |
| $ w_t $ | The **center word** (target word) to be predicted. |
| $ P(w_t \mid \text{context}) $ | The **conditional probability** of $ w_t $ being the correct word, given the context. |
| $ log P(w_t \mid \text{context}) $ | The **log-probability** of predicting $ w_t $ correctly given its context. |


---

#### Parameters Learned

- $ E \in {R}^{V \times d} $: Embedding matrix (input)
- $ E' \in {R}^{V \times d} $: Output matrix (used for prediction)

**Note**: Final embeddings are typically taken from matrix $ E $

---

### Summary

- **Input**: Integer indices of context words  
- **Output**: Predict index of target (center) word  
- **Lookup**: Embed each context word, then average  
- **Train**: With softmax or negative sampling  
- **Goal**: Words that appear in similar contexts should learn similar vector representations

---

### **Negative Sampling in Word2Vec**

---

#### Why Not Full Softmax?

In the original Skip-gram formulation, the softmax function is used to compute the probability of a context word given a target word:

$$
P(w_o \mid w_t) = \frac{\exp(u_{w_o}^\top v_{w_t})}{\sum_{w=1}^{V} \exp(u_w^\top v_{w_t})}
$$

Where:
- $ v_{w_t} $: input (center) embedding of word $ w_t $
- $ u_{w_o} $: output (context) embedding of word $ w_o $
- $ V $: vocabulary size

**Problem**:  
The denominator sums over **all words in the vocabulary** $ V $ — this is extremely slow for large corpora.

---

#### Solution: Negative Sampling

Instead of updating **all output weights**, **update only a small number of "negative samples"** along with the one positive pair.

---

#### Intuition

- For each training pair $ (w_t, w_o) $:
  - Treat it as a **positive example**
  - Sample $ k $ random words $ \{w_1^{'}, w_2^{'}, ..., w_k^{'}\} $ from the vocabulary — these are **negative samples**
- The goal:
  - **Maximize** the probability of real context word $ w_o $
  - **Minimize** the probability of negative samples $ w_i^{'} $

---

#### Loss Function

For a center word $ w_t $ and a context word $ w_o $, the **negative sampling loss** is:

$$
{L}_{\text{negative-sampling}} = -\log \sigma(u_{w_o}^\top v_{w_t}) - \sum_{i=1}^{k} \log \sigma(-u_{w_i^{'}}^\top v_{w_t})
$$

Where:
- $ \sigma(x) = \frac{1}{1 + \exp(-x)} $  : sigmoid function
- $ u_w $: output embedding  of word $ w $
- $ v_w $: input embedding  of word $ w $
- $ w_o $: actual word (positive sample)
- $ w_i^{'} $: i-th negative sample
- $ k $: number of negative samples

---

#### Training Workflow (Step-by-Step)

1. **Input**: target word $ w_t $, context word $ w_o $
2. **Get embeddings**:
   - $ v_{w_t} \in {R}^d $ from input matrix $ E $
   - $ u_{w_o} \in {R}^d $ from output matrix $ E'$
3. **Sample** $ k $ random words from vocabulary as negative samples
4. **Compute loss** using the formula above
5. **Backpropagate** and update only:
   - $ v_{w_t} $
   - $ u_{w_o} $
   - $ u_{w_i^{'}} $ for all $ i \in [1, k] $

---




#### Why It Works

- Drastically reduces **computation time**.
- Still allows the model to learn **good semantic embeddings**.
- Scales to **billions of tokens** and **millions of words**.
- Enables **mini-batch training** with sparse updates.


---
### **Hierarchical Softmax in Word2Vec**

---

####  Why Another Alternative to Softmax?

Like Negative Sampling, **Hierarchical Softmax (HS)** addresses the computational inefficiency of the full softmax, especially for large vocabularies.

**Key Idea**:  
 Instead of scoring all words in the vocabulary, we build a **binary tree** and compute the probability of a word by traversing the path from the **root to that word's leaf**.

---

#### How Hierarchical Softmax Works

1. Build a **binary tree** where:
   - **Leaves** represent vocabulary words.
   - **Internal nodes** make **binary decisions**: left/right.

2. Each word is uniquely identified by a **path from the root to a leaf**.

3. The model:
   - Learns vector representations for each **internal node**.
   - Predicts the correct **binary decision** at each node in the path.

---

#### Example

Say the word **"sat"** is located along this binary path:
```
Root → Left → Right → Left → [sat]
```

We must predict:
- Step 1: Go **Left**
- Step 2: Go **Right**
- Step 3: Go **Left**

So the model predicts **a sequence of decisions** — **not** the word directly.

---

#### Mathematical Formulation

Let:
- $ w $ : target word to predict
- $ w_t $ : input word
- $ L(w) $ : length of the binary path to $ w $
- $ n_i $ : the $ i^{th} $ internal node along the path to $ w $
- $ s_i \in \{+1, -1\} $: direction at node $ n_i $ (+1 = left, -1 = right)
- $ v_{w_t} $ : input vector for $ w_t $
- $ v_{n_i} $ : vector associated with internal node $ n_i $
- $ \sigma(x) = \frac{1}{1 + \exp(-x)} $

Then:

$$
P(w \mid w_t) = \prod_{i=1}^{L(w)} \sigma\left( s_i \cdot v_{n_i}^\top v_{w_t} \right)
$$

---

#### Loss Function

For predicting word $ w $ given center word $ w_t $, the **loss is**:

$$
{L}_{\text{HS}} = - \sum_{i=1}^{L(w)} \log \sigma\left( s_i \cdot v_{n_i}^\top v_{w_t} \right)
$$

- Similar to Negative Sampling, the model uses **sigmoid** activations at each node.
- Each decision is treated as a **binary classification task**.

---


#### Summary

- Hierarchical Softmax is a tree-based approximation to full softmax.
- Words are predicted by following binary paths.
- Each step in the path is trained via sigmoid + binary cross-entropy.
- Enables fast, full-vocab prediction with logarithmic complexity.

---



## **2. Task Explanation [Implementation - $200$ marks]**

### **Goal**:

- Implement **Word2Vec** from scratch in Python using **NumPy only** (no external ML/DL libraries such as PyTorch or TensorFlow).
- Use the below code to download the dataset for training and testing.
  ```python
  # Expectation-Maximization
  word2vec_train = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-word2vec", split="train")
  word2vec_val = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-word2vec", split="val")
  ```
- Support both **Skip-Gram** and **CBOW** architectures.
- Implement two training strategies for each:
  - **Negative Sampling**
  - **Hierarchical Softmax**

- After training your Word2Vec model, save the learned embeddings along with the vocabulary mappings using `pickle`:

  ```python
  import pickle

  with open("embeddings.pkl", "wb") as f:
      pickle.dump({
          "embeddings": embeddings,     # numpy  array of shape (vocab_size, embedding_dim)
          "vocab": vocab.itos,          # list: index → token
          "stoi": vocab.stoi            # dict: token → index
      }, f)
  ```

- Evaluate the learned embeddings using a **word analogy task** and generate a result CSV file with predictions from all four models.


**Important**: Every single component of the algorithm — forward pass, backward pass, gradient calculation, parameter updates, hierarchical softmax (Huffman Tree), negative sampling, and training loop — must be written from scratch **without using any external machine learning or deep learning libraries**.


---

### **Implementation Requirements**

#### **1. Preprocessing**
- Use `nltk.word_tokenize` for tokenization.
- Lowercase all tokens.
- Apply minimum frequency pruning: `min_count = 2`.
- Construct vocabulary such that the most frequent word gets index `0`, next most frequent gets `1`, and so on.

#### **2. Training Configuration**
- Embedding Dimension: `100`
- Sliding Window Size: `5`
- Random Seed: `42` (ensure deterministic initialization for reproducibility)
- Architecture Modes:  
  - `Skip-Gram with Negative Sampling`  
  - `Skip-Gram with Hierarchical Softmax`  
  - `CBOW with Negative Sampling`  
  - `CBOW with Hierarchical Softmax`  


#### **3. You MUST implement the following from scratch:**
- Vocabulary construction with frequency-based indexing
- Forward pass using dot product between input and output embeddings
- Backward pass with gradient computation and update rules
- **Negative Sampling**:
  - Use a noise distribution proportional to

     $$
    \text{Unigram}^{0.75}
     $$


  - Sample negative examples independently
- **Hierarchical Softmax**:
  - Construct a **Huffman Tree** based on word frequencies
  - Compute paths and binary codes for each word
  - Implement internal node updates and loss calculations
- Optimization using Mini Bacth stochastic gradient descent (SGD)



### **3. Analogy Evaluation**

Use the provided **analogy dataset** where each line is of the format:

```
word1 + word2 - word3 = ?
```

- For each trained model, compute:
  

  $$
  \vec = \text{embedding}[\text{word1}] + \text{embedding}[\text{word2}] - \text{embedding}[\text{word3}]
  $$
- Predict the closest word in embedding space to `vec` (excluding word1, word2, word3).
- Save predictions in a CSV with the following format:

```
word1,op1,word2,op2,word3,skipgram_ns_word,skipgram_hs_word,cbow_ns_word,cbow_hs_word
```


---




### **4. Semantic Neighborhood Visualization using t-SNE and UMAP (with Plotly)**

In this section, you will explore the **semantic structure of your learned embeddings** by visualizing neighborhoods of similar words using t-SNE and UMAP.

---

####  **Objective**  
Visualize semantically similar clusters by projecting 500 word vectors into 2D space using t-SNE and UMAP.

---

####  Architectures to visualize:
- CBOW + Negative Sampling
- CBOW + Hierarchical Softmax
- Skip-gram + Negative Sampling
- Skip-gram + Hierarchical Softmax

---

####  **Steps to Follow**

1. **Randomly Sample 50 Words**
   - From your vocabulary, randomly select 50 unique words that:
     - Are not special tokens like `<unk>` or `<pad>`

2. **Find Top 10 Most Similar Words (Cosine Similarity)**
   - For each of the 50 sampled words:
     - Compute cosine similarity with **all words in the vocabulary**
     - Select the top 10 most similar words (excluding the word itself)
   - Total words = 50 × 10 = **500 word vectors**

3. **Extract Embeddings**
   - Collect the embeddings for all 500 words.
   - Store the word labels for plotting.

4. **t-SNE and UMAP Reduction**
   - Apply **t-SNE** and **UMAP** to project the 500 embeddings to 2D.
   - Use:
     - `sklearn.manifold.TSNE(n_components=2)`
     - `umap.UMAP(n_components=2)`

5. **Visualize using Plotly**
   - Create **interactive scatter plots** using `plotly.express.scatter`
   - Include:
     - Word label as hover tooltip
     - Title as model type + method (e.g., `t-SNE | Skip-gram + HS`)
     - Save as `.html`

6. **Save Output**
   - Save the plots with descriptive names like:
     - `skipgram_ns_tsne.html`
     - `skipgram_ns_umap.html`

---

### **5. Deliverables**

- **Embeddings**: Save the final word embeddings as four `.pkl` files:
  - `skipgram_ns.pkl`
  - `skipgram_hs.pkl`
  - `cbow_ns.pkl`
  - `cbow_hs.pkl`
  

- **Results**: Save the analogy task predictions in a CSV file:
  - `word2vec_analogy_results.csv`


- **Plots** : 8 Plots of T-sne and UMAP
  - `skipgram_ns_tsne.html`
  - `skipgram_ns_umap.html`

  Similar naming convention for the other plots.


---
**Note**: This assignment is a way to explore various trajectories for a given problem. Clarifying every single minute detail about the implementation like hyperparameters, tolerance limit for early stopping etc. will not be entertained on Discord. You can always explore multiple paths and select the most suitable solution for the assignment. You can make assumptions about the implementation details and document it in the code. It will be highly rewarded.

---

### **Additional Suggestions to Fix for Determinism & Reproducibility**

- Use `np.random.seed(42)` globally for all randomness (weight init, sampling).

---

### **Operating Constraints**

- DO NOT use libraries like PyTorch, TensorFlow, Gensim.
- ONLY use **Python standard library**  , **NumPy** , **Pandas**  , **Plotly** ,  **scikit-learn** .

- Ensure clear, well-commented code and separate training routines for each mode.

---

In [ ]:
import nltk
import os
from datasets import load_dataset

from typing import List  # Import List from typing
from tqdm import tqdm  # Import tqdm for progress bar
from nltk.tokenize import word_tokenize  # Import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

hf_token = ""


# Config

EMBEDDING_DIM = 100
MIN_COUNT = 5
WINDOW_SIZE = 2
SEED = 42

# Loading Data
def load_hf_data() -> List[List[str]]:
    print("Loading and tokenizing dataset...")
    ds = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-word2vec", token=hf_token, split="train",download_mode='force_redownload')
    text_data = ds["text"]
    return [word_tokenize(doc.strip().lower()) for doc in tqdm(text_data, desc="Tokenizing Sentences : ") if doc.strip()]


data = load_hf_data()


Loading and tokenizing dataset...


README.md:   0%|          | 0.00/6.20k [00:00<?, ?B/s]

Assignment-3/word2vec/train.parquet:   0%|          | 0.00/34.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Tokenizing Sentences : 100%|██████████| 200000/200000 [01:44<00:00, 1911.32it/s]


In [ ]:
#LIBRARIES
from tqdm import tqdm
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import re
import pickle
import heapq
import math
import random
np.random.seed(42)




In [ ]:
#TEST CELL
import numpy as np
list_2d = np.array(data, dtype=object)

sample_trainds = list_2d[:2]

In [ ]:
#CBOW with negative sampling

np.random.seed(42)

#preprocessing--- 1)Remove predefined word. 2)convert into lower case
def preprocess_text(list_2d):
    processed_data = []
    words_to_remove = {'unk', 'pad', '<unk>', '<pad>', 'UNK', 'PAD'}
    for sublist in list_2d:
        cleaned_sublist = [
            re.sub(r'[^\w\s]', '', token.lower())
            for token in sublist
            if (str(token) is not None and
                str(token).strip().lower() not in words_to_remove)
        ]
        cleaned_sublist = [token for token in cleaned_sublist if token]
        if cleaned_sublist:
            processed_data.append(cleaned_sublist)
    return processed_data
#-----


#Build vocab--- 1)Iterate over data and update the count of every word
def build_vocab(processed_data):
    word_counts = Counter()
    for sentence in processed_data:
        word_counts.update(sentence)

    vocab = {"<unk>": 0}
    for idx, (word, _) in enumerate(word_counts.most_common(), start=1):
        vocab[word] = idx

    return vocab, len(vocab), word_counts #it is returning the counter of the all unique word i.e Vocab, len and word_count


def stable_sigmoid(x):# Sigmoid function used as activatin function
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))

def probabilities(word_counter, vocab, V):#used as described in the Task to use this probability distribution
    word_freqs = np.zeros(V, dtype=np.float64)

    for word, count in word_counter.items():
        idx = vocab[word]
        word_freqs[idx] = count

    word_freqs[vocab["<unk>"]] = 1

    word_probs = word_freqs ** 0.75 #unigram ^^ 0.75
    word_probs /= word_probs.sum()
    return word_probs ##returning the probability distribution

def get_context_vectorized(sentences, vocab, WINDOW_SIZE=2): #find the vector of the context word using the window size
    all_targets, all_contexts = [], []

    for sentence in sentences:
        sentence_idx = [vocab.get(word, vocab["<unk>"]) for word in sentence] #get the word from the vocab if the word is not present in the vocab
                                                                              #than fetch the <unk>. <unk> is used to handle the OOV problem.
        #After this for loop sentence_idx contain the word and its index as used in the vocab dictionary

        for i in range(len(sentence_idx)):
            target = sentence_idx[i]
            start, end = max(0, i - WINDOW_SIZE), min(len(sentence_idx), i + WINDOW_SIZE + 1) #this make sure that window don't get outside the sentence_idx
            context = [sentence_idx[j] for j in range(start, end) if j != i]

            if context:
                all_targets.append(target)
                all_contexts.append(context)
          #all_target contains the target word && all_contexts contains the context word which will be used as input to predict target word.
    return np.array(all_targets), all_contexts

def train_cbow_negative_sampling(
    corpus, epochs=10, lr=0.05, k=5, WINDOW_SIZE=2, DIMENSION=100, batch_size=512
):
    # Preprocess
    cleaned_data = preprocess_text(corpus)
    vocab, V, word_counter = build_vocab(cleaned_data)
    word_probs = probabilities(word_counter,vocab, V)

    # Initialize weights
    W_in = np.random.randn(V, DIMENSION) * 0.01
    W_out = np.random.randn(V, DIMENSION) * 0.01
    # Initialize wts with random very small Number

    # Training data
    targets, contexts_list = get_context_vectorized(cleaned_data, vocab, WINDOW_SIZE)
    #gets the target and its context vectorized vector as describe above

    N = len(targets)

    for epoch in range(epochs):
        indices = np.arange(N)
        np.random.shuffle(indices)

        total_loss = 0.0

        for b in tqdm(range(0, N, batch_size), desc=f"Epoch {epoch+1}/{epochs}"):
            batch_idx = indices[b:b+batch_size]
            t_batch = targets[batch_idx]  # contins the target words indices for this batch
            c_batch = [contexts_list[i] for i in batch_idx]
            #Now making batch of 512 sentence for speed up

            # Instead of usin one-hot vector we are using word indices and then avg them and give it as hidden layer
            # By doing so we are avoiding sparse matrix

            max_len = max(len(ctx) for ctx in c_batch)# Used to declared the matrix size
            mask = np.zeros((len(c_batch), max_len), dtype=int)  #--> This creates a matrix where each row represents a context with word indices
            for i, ctx in enumerate(c_batch):                    # SIze of the matrx is (batch size, max content size in this batch)
                mask[i, :len(ctx)] = ctx
            ctx_embeds = W_in[mask]
            h_batch = np.mean(ctx_embeds, axis=1)

            #for +ve samples
            u_pos = W_out[t_batch] # u_pos contains the wt of the target word indices
            score_pos = np.sum(h_batch * u_pos, axis=1)
            pred_pos = stable_sigmoid(score_pos)

            #for -ve samples
            neg_samples = np.random.choice(V, size=(len(t_batch), k), p=word_probs) #samples k negative words for each in the batch
            u_negs = W_out[neg_samples]               #embeddings for the negative samples lookup into W_out
            score_negs = np.einsum("bd,bkd->bk", h_batch, u_negs) # for each batch, multiply (D,) context vector with (k, D) negative vectors
            pred_negs = stable_sigmoid(-score_negs)

            #calculate loss
            loss = -np.log(np.clip(pred_pos, 1e-10, 1.0)) \
                   - np.sum(np.log(np.clip(pred_negs, 1e-10, 1.0)), axis=1)
            total_loss += np.sum(loss)

            # finding gradient
            grad_pos_output = (pred_pos - 1)[:, None] * h_batch #Gradient for positive output embeddings
            grad_negs_output = (1 - pred_negs)[:, :, None] * h_batch[:, None, :] #Gradient for negative output embeddings

            grad_h = (pred_pos - 1)[:, None] * u_pos + np.einsum("bk,bkd->bd", (1 - pred_negs), u_negs) #Gradient for hidden layer that will be backpropagated to input embeddings W_in)

            # Updates
            W_out[t_batch] -= lr * grad_pos_output
            for i in range(k):
                np.add.at(W_out, neg_samples[:, i], -lr * grad_negs_output[:, i, :])

            # Vectorized update for input embeddings
            for i, ctx in enumerate(c_batch):
                if len(ctx) > 0:
                    W_in[ctx] -= lr * grad_h[i] / len(ctx)

        avg_loss = total_loss / N
        print(f"Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")

    # Save embeddings
    with open("CBOW_negative_sampling_embeddings_vectorized.pkl", "wb") as f:
        pickle.dump({
            "embeddings": W_in,
            "stoi": vocab,
            "itos": {i: w for w, i in vocab.items()},
        }, f)

    return W_in, vocab


array_2d = np.array(data, dtype=object)
W_in, vocab = train_cbow_negative_sampling(array_2d)

# processed_data = preprocess_text(list_2d)


In [ ]:
#CBOW with hierarchical softmax

class HuffmanNode:
    def __init__(self, freq, index=None, left=None, right=None):
        self.freq = freq
        self.index = index
        self.left = left
        self.right = right
    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(freqs): #Builds huffman tree that ensure that frequent word take smaller path
    heap = []
    nodes = []
    for i, f in enumerate(freqs):
        node = HuffmanNode(f, index=i)
        heapq.heappush(heap, node)
        nodes.append(node)
        # in this while loop combining two least frequent word to make a single loop
    while len(heap) > 1:
        a = heapq.heappop(heap)
        b = heapq.heappop(heap)
        parent = HuffmanNode(a.freq + b.freq, left=a, right=b)
        heapq.heappush(heap, parent)
        nodes.append(parent)
    root = heap[0]
    codebook = {}# it maintain the binary code for each word
    inner_nodes = []
    def dfs(node, code_bits, inner_path):
        if node.index is not None:
            codebook[node.index] = (code_bits.copy(), inner_path.copy())
            return
        node_id = len(inner_nodes)
        inner_nodes.append(node)
        dfs(node.left, code_bits + [0], inner_path + [node_id])
        dfs(node.right, code_bits + [1], inner_path + [node_id])

        # this is a special case when we are having only one word in the vocab
    if root.index is not None:
        codebook[root.index] = ([0], [])
        return root, codebook, 0
    dfs(root, [], [])
    return root, codebook, len(inner_nodes)


def preprocess_text(list_2d):
  """Preprocess the 2D list of tokens"""
  processed_data = []
  words_to_remove = {'unk', 'pad', '<unk>', '<pad>', 'UNK', 'PAD'}
  for sublist in list_2d:
      # Clean each token: lowercase, remove punctuation
      cleaned_sublist = [
          re.sub(r'[^\w\s]', '', token.lower())
          for token in sublist
          if (str(token) is not None and
              str(token).strip().lower() not in words_to_remove)
      ]
      # Remove empty strings
      cleaned_sublist = [token for token in cleaned_sublist if token]
      if cleaned_sublist:  # Only add non-empty lists
          processed_data.append(cleaned_sublist)
  return processed_data

def build_vocab(processed_data):#Build vocab similar to cbow with negative sampling
  word_counts = Counter()
  for Sentences in processed_data:
    word_counts.update(Sentences)

  vocab = {word: count for word, count in word_counts.items() if count >= 5}
  index2word = list(vocab.keys())
  word2index = {word: idx for idx, word in enumerate(index2word)}
  freqs = np.array([vocab[w] for w in index2word], dtype=np.float32)
  return vocab, index2word, word2index, freqs


def initialize_embedding(V, dim, n_inner):  # word embedding are initialize with very small random no.
  W_in = (np.random.rand(V, dim) - 0.5) / dim
  W_out = (np.random.rand(max(1, n_inner), dim) - 0.5) / dim
  return W_in, W_out

def context_vector(W_in, context_idx): # Return the avg of the context vector
  return np.mean(W_in[context_idx], axis=0)

def sigmoid(x):  # To find the sigmoid value
  return 1 / (1+np.exp(-x))

def cbow_with_hierarchical_sfmx(corpus, dim=100, window=2, epochs=10, lr=0.05):

  cleaned_data = preprocess_text(corpus)
  vocab, index2word, word2index, freqs = build_vocab(cleaned_data)
  V = len(index2word)
  _, codebook, n_inner = build_huffman_tree(freqs.tolist())


  W_in, W_out = initialize_embedding(V, dim, n_inner)


  #Build huffman tree of vocab using the word_counter

  for epoch in range(0, epochs):
    loss_epoch = 0
    word_count = 0

    for sent in tqdm(cleaned_data, desc=f"Epoch {epoch+1}/{epochs}", unit="sent"):
      L = len(sent)
      for pos, target_word in enumerate(sent):
        if target_word not in word2index:
          continue
        target_idx = word2index[target_word]
        # finding context word around our target words
        context_indices = []
        start = max(0, pos-window)
        end = min(L, pos+window+1)
        for i in range(start, end):
          if i == pos: continue # to skip the target word because start and end contains the target word

          w = sent[i]
          if w in word2index:
            context_indices.append(word2index[w])
        if len(context_indices) == 0:
          continue

       #forward pass and get the avg of context and word embeddings
        h = context_vector(W_in, context_indices)
        # Looking for the huffman code for our target word
        bits, node_ids = codebook[target_idx]
        if len(node_ids) == 0:
          continue


        u = W_out[node_ids]                      # shape (path_len, dim)
        scores = u @ h                           # (path_len,) dot product
        sigmas = sigmoid(scores)                 # (path_len,) probabilities
        gs = (sigmas - np.array(bits))           # (path_len,) how much wrong we are

        #From here onward we are calculating how much wrong we are with our predicition
        loss_node = 0
        for sigma, bit in zip(sigmas, bits):
          if bit == 1:
            loss_node += -math.log(max(sigma, 1e-12))   # target = 1 meaning want our prob. near 1
          else:
            loss_node += -math.log(max(1 - sigma, 1e-12))  # target = 0 in this case want our prob. near 0
          loss_epoch += loss_node

        # Update W_out (vectorized)
        W_out[node_ids] -= lr * gs[:, None] * h  # broadcasting

        # Gradient for h
        grad_h = np.sum(gs[:, None] * u, axis=0)




        # coef = 1.0/len(context_indices)
        # for idx in context_indices:
        #   W_in[idx] -= lr*coef*grad_h
        W_in[context_indices] -= (lr / len(context_indices)) * grad_h


        word_count += 1
    print(f"Epoch {epoch}/{epochs} Loss: {loss_epoch:.4f} Processed words: {word_count}")
  return W_in, W_out, index2word, word2index


W_in, W_out, index2word, word2index = cbow_with_hierarchical_sfmx(list_2d, dim=100, window=2, epochs=5, lr=0.05)

with open("cbow_hierarchical_softmax.pkl", "wb") as f:
    pickle.dump({
        "embeddings": W_in,      # numpy array of shape (vocab_size, embedding_dim)
        "vocab": index2word,     # list: index -> token
        "stoi": word2index       # dict: token -> index
    }, f)





Epoch 1/5: 100%|██████████| 129005/129005 [20:09<00:00, 106.67sent/s]


Epoch 0/5 Loss: 448422218.6177 Processed words: 9316806


Epoch 2/5: 100%|██████████| 129005/129005 [19:48<00:00, 108.57sent/s]


Epoch 1/5 Loss: 437288552.7463 Processed words: 9316806


Epoch 3/5: 100%|██████████| 129005/129005 [19:36<00:00, 109.61sent/s]


Epoch 2/5 Loss: 432816963.2157 Processed words: 9316806


Epoch 4/5: 100%|██████████| 129005/129005 [19:57<00:00, 107.70sent/s]


Epoch 3/5 Loss: 430073016.7399 Processed words: 9316806


Epoch 5/5: 100%|██████████| 129005/129005 [19:54<00:00, 108.04sent/s]


Epoch 4/5 Loss: 428134513.1182 Processed words: 9316806


In [ ]:
# Skipgram with Negative Sampling

# Preprocessing just as done above
def preprocess_text(list_2d):
    processed_data = []
    words_to_remove = {'unk', 'pad', '<unk>', '<pad>', 'UNK', 'PAD'}
    for sublist in list_2d:
        cleaned_sublist = [
            re.sub(r'[^\w\s]', '', token.lower())
            for token in sublist
            if token and token.lower() not in words_to_remove
        ]
        cleaned_sublist = [token for token in cleaned_sublist if token]
        if cleaned_sublist:
            processed_data.append(cleaned_sublist)
    return processed_data
# similar function as in cbow
def build_vocab(processed_data):
    word_counts = Counter()
    for sentence in processed_data:
        word_counts.update(sentence)
    vocab = {word: idx for idx, (word, _) in enumerate(word_counts.most_common())}
    return vocab, len(vocab), word_counts

# Similar initializing embedding
def initialize_embedding(V, DIMENSION):
    W_in = np.random.randn(V, DIMENSION) * 0.01
    W_out = np.random.randn(V, DIMENSION) * 0.01
    return W_in, W_out

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def probabilities(word_counter):
    word_freqs = np.array([count for _, count in word_counter.items()], dtype=np.float32)
    word_probs = word_freqs ** 0.75
    word_probs /= word_probs.sum()
    return word_probs


def get_skipgram_pairs(sentences, vocab, WINDOW_SIZE=2):
    targets, contexts = [], []
    for sentence in sentences:
        sentence_idx = [vocab[w] for w in sentence if w in vocab]
        for i, target in enumerate(sentence_idx):
          #star and end are pointer of the window
            start, end = max(0, i - WINDOW_SIZE), min(len(sentence_idx), i + WINDOW_SIZE + 1)
            for j in range(start, end):
                if j != i: # to avoid taking the word itself
                    targets.append(target)
                    contexts.append(sentence_idx[j])
    return np.array(targets), np.array(contexts)

# Trainig phase
def train_skipgram_negative_sampling(
    corpus, epochs=5, lr=0.05, k=5, WINDOW_SIZE=2, DIMENSION=50, batch_size=512
):
    # Preprocess
    cleaned_data = preprocess_text(corpus)
    vocab, V, word_counter = build_vocab(cleaned_data)
    word_probs = probabilities(word_counter)

    # Init embeddings
    W_in, W_out = initialize_embedding(V, DIMENSION)

    # Generate training pairs once
    targets, contexts = get_skipgram_pairs(cleaned_data, vocab, WINDOW_SIZE)
    N = len(targets)

    for epoch in range(epochs):
        # Shuffle indices so that we don't learn the order
        indices = np.arange(N)
        np.random.shuffle(indices)

        total_loss = 0.0
        # Process data in batches to speed thing up

        for b in tqdm(range(0, N, batch_size), desc=f"Epoch {epoch+1}/{epochs}"):
            batch_idx = indices[b:b+batch_size]
            t_batch, c_batch = targets[batch_idx], contexts[batch_idx]

            # Vectors for the center word and their actual neighbours
            v_inputs = W_in[t_batch]          # (B, D)
            u_pos = W_out[c_batch]            # (B, D)

            # how similar each center word is to its real context
            score_pos = np.sum(v_inputs * u_pos, axis=1)      # (B,)
            pred_pos = sigmoid(score_pos)                     # (B,)

            # Randomly selected the negative samples from the possible output to speed up the process
            neg_samples = np.random.choice(V, size=(len(t_batch), k), p=word_probs)
            u_negs = W_out[neg_samples]       # (B, k, D) -> dimension for random words

            # Negative scores
            score_negs = np.einsum("bd,bkd->bk", v_inputs, u_negs)   # (B, k)
            pred_negs = sigmoid(-score_negs)                         # (B, k)

            # calculating Loss
            loss = -np.log(pred_pos + 1e-10) - np.sum(np.log(pred_negs + 1e-10), axis=1)
            total_loss += np.sum(loss)

            # Gradients to adjust vector for better predictions
            grad_input = (pred_pos - 1)[:, None] * u_pos + np.einsum(
                "bk,bkd->bd", (1 - pred_negs), u_negs
            )

            grad_pos = (pred_pos - 1)[:, None] * v_inputs
            grad_negs = (1 - pred_negs)[:, :, None] * v_inputs[:, None, :]

            # Update embeddings based on the error
            W_in[t_batch] -= lr * grad_input
            W_out[c_batch] -= lr * grad_pos
            np.add.at(W_out, neg_samples, -lr * grad_negs)

        avg_loss = total_loss / N
        print(f"Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")

    # Save embeddings
    with open("skipgram_negative_sampling_embeddings.pkl", "wb") as f:
        pickle.dump({
            "embeddings": W_in,
            "vocab": list(vocab.keys()),
            "stoi": vocab
        }, f)

    return W_in, vocab
W_in, vocab = train_skipgram_negative_sampling(list_2d, epochs=10, lr=0.05, k=5, WINDOW_SIZE=2, DIMENSION=100, batch_size=512)

Epoch 1/10: 100%|██████████| 72454/72454 [14:03<00:00, 85.89it/s]


Epoch 1, Average Loss: 1.7078


Epoch 2/10: 100%|██████████| 72454/72454 [13:58<00:00, 86.42it/s]


Epoch 2, Average Loss: 1.5757


Epoch 3/10: 100%|██████████| 72454/72454 [13:52<00:00, 87.05it/s]


...

Epoch 8, Average Loss: 1.5055


Epoch 9/10: 100%|██████████| 72454/72454 [14:03<00:00, 85.87it/s]


Epoch 9, Average Loss: 1.5021


Epoch 10/10: 100%|██████████| 72454/72454 [13:59<00:00, 86.28it/s]


Epoch 10, Average Loss: 1.4988


In [ ]:
import heapq
import re
import numpy as np
import math
from collections import Counter
from tqdm import tqdm
import pickle

#Skip gram with Hierarchical Softmax

class HuffmanNode:
    def __init__(self, freq, index=None, left=None, right=None):
        self.freq = freq
        self.index = index
        self.left = left
        self.right = right
    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(freqs):
    heap, nodes = [], []
    for i, f in enumerate(freqs):
        node = HuffmanNode(f, index=i)
        heapq.heappush(heap, node)
        nodes.append(node)
    while len(heap) > 1:
        a, b = heapq.heappop(heap), heapq.heappop(heap)
        parent = HuffmanNode(a.freq + b.freq, left=a, right=b)
        heapq.heappush(heap, parent)
        nodes.append(parent)
    root = heap[0]
    codebook, inner_nodes = {}, []

    def dfs(node, code_bits, inner_path):
        if node.index is not None:
            codebook[node.index] = (np.array(code_bits, dtype=np.int8),
                                    np.array(inner_path, dtype=np.int32))
            return
        node_id = len(inner_nodes)
        inner_nodes.append(node)
        dfs(node.left, code_bits + [0], inner_path + [node_id])
        dfs(node.right, code_bits + [1], inner_path + [node_id])

    if root.index is not None:
        codebook[root.index] = (np.array([0], dtype=np.int8), np.array([], dtype=np.int32))
        return root, codebook, 0
    dfs(root, [], [])
    return root, codebook, len(inner_nodes)

def preprocess_text(list_2d):
    words_to_remove = {'unk', 'pad', '<unk>', '<pad>', 'UNK', 'PAD'}
    processed_data = []
    for sublist in list_2d:
        cleaned = [
            re.sub(r'[^\w\s]', '', token.lower())
            for token in sublist
            if token and str(token).lower() not in words_to_remove
        ]
        cleaned = [w for w in cleaned if w]
        if cleaned:
            processed_data.append(cleaned)
    return processed_data

def build_vocab(processed_data, min_count=5):
    word_counts = Counter()
    for sent in processed_data:
        word_counts.update(sent)
    vocab = {w:c for w,c in word_counts.items() if c >= min_count}
    index2word = list(vocab.keys())
    word2index = {w:i for i,w in enumerate(index2word)}
    freqs = np.array([vocab[w] for w in index2word], dtype=np.float32)
    return vocab, index2word, word2index, freqs

def initialize_embedding(V, dim, n_inner):
    W_in = (np.random.rand(V, dim) - 0.5) / dim
    W_out = (np.random.rand(max(1, n_inner), dim) - 0.5) / dim
    return W_in, W_out

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -10, 10)))  # Added clipping for stability


#build huffman tree, dfs, preprocess_text, build vocab , initialize embedding, sigmoid function
# all this function are similar to the previous task

def skipgram_with_hierarchical_softmax(corpus, dim=100, window=2, epochs=5, lr=0.025):
    cleaned_data = preprocess_text(corpus)
    vocab, index2word, word2index, freqs = build_vocab(cleaned_data, min_count=5)  #use min_count=5
    V = len(index2word)

    if V == 0:
        raise ValueError("Vocabulary is empty. Check your data and min_count.")

    _, codebook, n_inner = build_huffman_tree(freqs.tolist())
    W_in, W_out = initialize_embedding(V, dim, n_inner)

    for epoch in range(epochs):

      total_loss, word_count = 0, 0
      for sent in tqdm(cleaned_data, desc=f"Epoch {epoch+1}/{epochs}", unit="sent"):
          L = len(sent)
          for pos, center in enumerate(sent):
              if center not in word2index:
                  continue
              c_idx = word2index[center]
              v_center = W_in[c_idx].copy()  # Make a copy to accumulate gradients

              # Collect all context indices
              start, end = max(0, pos - window), min(L, pos + window + 1)
              context_indices = []
              for i in range(start, end):
                  if i == pos:
                      continue
                  w = sent[i]
                  if w in word2index:
                      context_indices.append(word2index[w])

              if not context_indices:
                  continue

              # Process each context word
              grad_v_center_total = np.zeros_like(v_center)

              for ctx_idx in context_indices:
                  bits, node_ids = codebook[ctx_idx]
                  if node_ids.size == 0:
                      continue

                  u_nodes = W_out[node_ids]  # (path_len, dim)
                  scores = u_nodes @ v_center
                  sigmas = sigmoid(scores)
                  gs = sigmas - bits

                  # Calculate loss
                  total_loss += -np.sum(bits * np.log(sigmas + 1e-12) +
                                      (1 - bits) * np.log(1 - sigmas + 1e-12))

                  # Update W_out
                  W_out[node_ids] -= lr * gs[:, None] * v_center

                  # Accumulate gradient for center word
                  grad_v_center_total += np.sum(gs[:, None] * u_nodes, axis=0)

                  word_count += 1

              # Update center word embedding once for all context words
              W_in[c_idx] -= lr * grad_v_center_total

      print(f"Epoch {epoch+1}/{epochs} Loss={total_loss:.2f}  Pairs={word_count}")
      if epoch+1 in {3,5}:
        fname = f"skipgram_epoch{epoch+1}.pkl"
        with open(fname, "wb") as f:
          pickle.dump({
            "embeddings": W_in,
            "vocab": index2word,
            "stoi": word2index
          }, f)
    # Save embeddings (moved outside the function)
    return W_in, W_out, index2word, word2index


W_in, W_out, index2word, word2index = skipgram_with_hierarchical_softmax(
    list_2d, dim=100, window=2, epochs=5, lr=0.025
)


Epoch 1/5: 100%|██████████| 129005/129005 [39:52<00:00, 53.92sent/s]


Epoch 1/5 Loss=251107727.03  Pairs=36114676


Epoch 2/5: 100%|██████████| 129005/129005 [39:47<00:00, 54.02sent/s]


Epoch 2/5 Loss=245396766.82  Pairs=36114676


Epoch 3/5: 100%|██████████| 129005/129005 [39:57<00:00, 53.82sent/s]


Epoch 3/5 Loss=243643935.02  Pairs=36114676


Epoch 4/5: 100%|██████████| 129005/129005 [39:57<00:00, 53.81sent/s]


Epoch 4/5 Loss=242709589.84  Pairs=36114676


Epoch 5/5: 100%|██████████| 129005/129005 [39:59<00:00, 53.76sent/s]


Epoch 5/5 Loss=242115400.17  Pairs=36114676


In [ ]:
word2vec_test = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-word2vec-analogy", token=hf_token, split="test")


Assignment-3/word2vec/test.parquet:   0%|          | 0.00/32.4k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
import pandas as pd
word2vec_df = word2vec_test.to_pandas()
print(word2vec_df.head())

      word1    word2          word3
0    rotate      uss        albania
1    humber   perils       lakewood
2  alphonso  åberget          socks
3    coiner     digs  distributions
4   charted  merging        singles


In [ ]:
with open('cbow_ns.pkl', "rb") as f:
  data =pickle.load(f)
  stoi = data["stoi"]
  embedding = data["embeddings"]
  if "<unk>" in stoi:
    unk_idx = stoi["<unk>"]
    unk_embedding = embedding[unk_idx]

In [ ]:
with open("cbow_hs.pkl", "rb") as f:
    data = pickle.load(f)

# Extract embeddings and vocabulary
embeddings_cbow_hs = data["embeddings"]   # numpy array of shape (V, dim)
itos_cbow_hs = data["vocab"]
stoi_cbow_hs = data["stoi"]               # word -> index dictionary


with open("skipgram_ns.pkl", "rb") as f:
    data = pickle.load(f)

# Extract embeddings and vocabulary
embeddings_skipgram_ns = data["embeddings"]   # numpy array of shape (V, dim)
itos_skipgram_ns = data["vocab"]
stoi_skipgram_ns = data["stoi"]               # word -> index dictionary


with open("skipgram_hs.pkl", "rb") as f:
    data = pickle.load(f)

# Extract embeddings and vocabulary
embeddings_skipgram_hs = data["embeddings"]   # numpy array of shape (V, dim)
itos_skipgram_hs = data["vocab"]
stoi_skipgram_hs = data["stoi"]               # word -> index dictionary



with open("cbow_ns.pkl", "rb") as f:
    data = pickle.load(f)
    embeddings_cbow_ns = data["embeddings"]   # numpy array of shape (V, dim)
    itos_cbow_ns = data["itos"]
    stoi_cbow_ns = data["stoi"]




In [ ]:
#################### FInal cOde ################
def load_embedding_cbow_ns(file_path):
  with open(file_path, "rb") as f:
    data = pickle.load(f)
  return data["embeddings"], data["itos"], data["stoi"]

def load_embeddings(file_path):
    """Load embeddings from a pickle file."""
    with open(file_path, "rb") as f:
        data = pickle.load(f)
    return data["embeddings"], data["vocab"], data["stoi"]

def cosine_similarity(vec1, vec2):
    """Compute cosine similarity safely."""
    dot = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1) + 1e-10
    norm2 = np.linalg.norm(vec2) + 1e-10
    return float(dot / (norm1 * norm2))


def find_analogy(word1, word2, word3, embeddings, itos, stoi, unk_embedding):
    """
    Solve analogy: word1 + word2 - word3 = ?
    Handles unknown words using the <unk> embedding.
    """
    # Get indices or fallback to <unk> vector
    idx1 = stoi[word1] if word1 in stoi else None
    idx2 = stoi[word2] if word2 in stoi else None
    idx3 = stoi[word3] if word3 in stoi else None

    vec1 = embeddings[idx1] if idx1 is not None else unk_embedding
    vec2 = embeddings[idx2] if idx2 is not None else unk_embedding
    vec3 = embeddings[idx3] if idx3 is not None else unk_embedding

    # Compute analogy vector and normalize
    analogy_vec = vec1 + vec2 - vec3
    analogy_vec /= (np.linalg.norm(analogy_vec) + 1e-10)

    # Vectorized cosine similarities
    sims = embeddings @ analogy_vec

    # Exclude known input words
    exclude = [i for i in [idx1, idx2, idx3] if i is not None]
    sims[exclude] = -np.inf

    # Best match
    best_idx = np.argmax(sims)
    return itos[best_idx]


op1 = '+'
op2 = '-'
results = []
for idx, row in word2vec_df.iterrows():
    word1, word2, word3 = row['word1'], row['word2'], row['word3']

    sg_ns_word = find_analogy(word1, word2, word3, embeddings_skipgram_ns, itos_skipgram_ns, stoi_skipgram_ns, unk_embedding)
    sg_hs_word = find_analogy(word1, word2, word3, embeddings_skipgram_hs, itos_skipgram_hs, stoi_skipgram_hs, unk_embedding)
    cbow_ns_word = find_analogy(word1, word2, word3, embeddings_cbow_ns, itos_cbow_ns, stoi_cbow_ns  ,unk_embedding)
    cbow_hs_word = find_analogy(word1, word2, word3, embeddings_cbow_hs, itos_cbow_hs, stoi_cbow_hs, unk_embedding)

    results.append([word1, op1, word2, op2, word3, sg_ns_word, sg_hs_word, cbow_ns_word, cbow_hs_word])

# Save to CSV
output_df = pd.DataFrame(results, columns=['word1','op1','word2','op2','word3',
                                            'skipgram_ns_word','skipgram_hs_word',
                                            'cbow_ns_word','cbow_hs_word'])
output_df.to_csv("analogy_predictions.csv", index=False)
print("CSV saved as analogy_predictions.csv")


In [ ]:
# import numpy as np
import random
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import umap.umap_ as umap
import plotly.express as px

# Randomly select 50 words from the vocab

def sample_words(vocab, num_samples=50):
    special_tokens = {"<unk>", "<pad>"}
    valid_vocab = [w for w in vocab if w not in special_tokens]
    return random.sample(valid_vocab, num_samples)


def sample_words_cbow_ns(itos, n=50):
    indices = random.sample(list(itos.keys()), n)
    return [itos[i] for i in indices]


#find top-k most similar words for each sample
def get_similar_words(sampled_words, embeddings, stoi, vocab, top_k=10):
    all_word_vecs = []
    all_labels = []

    for word in sampled_words:
        idx = stoi[word]
        vec = embeddings[idx].reshape(1, -1)

        # Cosine similarity with all vocab
        sims = cosine_similarity(vec, embeddings)[0]

        # Top k indices (excluding the word itself)
        sim_indices = np.argsort(-sims)[1: top_k + 1]

        # Store results
        for j in sim_indices:
            all_word_vecs.append(embeddings[j])
            all_labels.append(vocab[j])

    return np.array(all_word_vecs), all_labels

#dimensionality reduction using t-SNE and UMAP as describe in the problem statement
def reduce_and_plot(embeddings, labels, model_name):
    # t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    tsne_coords = tsne.fit_transform(embeddings)
    fig_tsne = px.scatter(
        x=tsne_coords[:,0], y=tsne_coords[:,1],
        text=labels, title=f"{model_name} | t-SNE"
    )
    fig_tsne.write_html(f"{model_name}_tsne.html")

    # UMAP
    reducer = umap.UMAP(n_components=2, random_state=42)
    umap_coords = reducer.fit_transform(embeddings)
    fig_umap = px.scatter(
        x=umap_coords[:,0], y=umap_coords[:,1],
        text=labels, title=f"{model_name} | UMAP"
    )
    fig_umap.write_html(f"{model_name}_umap.html")

    print(f"Saved plots: {model_name}_tsne.html, {model_name}_umap.html")

#RUnning for each model

sampled = sample_words(itos_cbow_hs)
vecs, labels = get_similar_words(sampled, embeddings_cbow_hs, stoi_cbow_hs, itos_cbow_hs)
reduce_and_plot(vecs, labels, "cbow_hs")

sampled = sample_words_cbow_ns(itos_cbow_ns)
vecs, labels = get_similar_words(sampled, embeddings_cbow_ns, stoi_cbow_ns, itos_cbow_ns)
reduce_and_plot(vecs, labels, "cbow_ns")

sampled = sample_words(itos_skipgram_hs)
vecs, labels = get_similar_words(sampled, embeddings_skipgram_hs, stoi_skipgram_hs, itos_skipgram_hs)
reduce_and_plot(vecs,labels, "skipgram_hs")

sampled = sample_words(itos_skipgram_ns)
vecs, labels = get_similar_words(sampled, embeddings_skipgram_ns, stoi_skipgram_ns, itos_skipgram_ns)
reduce_and_plot(vecs,labels, "skipgram_ns")


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Saved plots: cbow_ns_tsne.html, cbow_ns_umap.html


## **3. Analytical & Critical Thinking Questions [$6 \times 20 = 120$ marks]**
---

### **Q1. Skip-gram vs CBOW Trade-offs**
Given a highly imbalanced corpus where rare words appear very few times and common words dominate, which architecture (Skip-gram or CBOW) would you prefer and why?  Justify your answer using the mathematical objectives and data characteristics.

(<b>Ans:</b><br>
Prefer skip-gram for a highly imbalanced corpus where many words are rare and few words dominate.

Skip gram uses a center word to predict its context so even rare words get strong direct updates when they appear as center words. A single occurrence of a word produces several positives training pairs so the rare word's embedding receives multiple informative updates from each occurance.

Objective of skip gram:
$$
P(\text{center}\mid \text{context})\text{ OR} \\
P(w_t \mid w_{t-c}, ...,  w_{t-1},w_{t+1}, ..., w_{t+c}) \\
\text{c: Context window size}
$$

### **Q2. Impact of Negative Sampling Distribution**
If negative samples are drawn uniformly at random versus proportionally to word frequency, how would the embeddings differ?  
Which strategy do you think better captures semantic relationships, and why?

<b>Ans:</b><br>
Negatives samples drawn uniformly at random:<br>
$$
P_{n}(w) = \frac{1}{|V|}
$$
<ul>
  <li>Rare words get sampled as negatives as often as frequent words.</li>
  <li>Model does not learn to seperate center word from frequent noise words.</li>
  <li>Frequent word may not be pushed far enough from unrelated contexts</li>
  <li>Rare words may be pushed randomly</li>
</ul>
<hr>

Negative samples drawn proportionally to word frequency:
$$
P_{n}(w) \propto f(w)^{\alpha} \\
\text{where α = 0.75 }
$$
<ul>
  <li>More frequent words are sampled compare to rare words.</li>
  <li>Pushes frequent noise words away from inappropiate contexts.</li>
  <li>Rare words are less likely to be sampled as negative.</li>
  <li>Better capture meaningful relationships. </li>
</ul>

Therefore frequency-based sampling leads to embedding capture better semantics relationship


### **Q3. Analogy Task Evaluation**
Explain why this vector arithmetic works in the embedding space.  
What assumptions about word co-occurrence and semantic regularities does this rely on?  
Can you think of situations where analogy tasks might fail, even if embeddings are high quality?


<b>Ans:</b><br>
Assumptions:
<ul>
  <li>Distributional hypothesis </li>
  <li>Relations are consistent and linear </li>
  <li>Sufficient co-occurrence data of all words</li>
  <li>Single meaning per word </li>
</ul>

It will fails:
<ul>
  <li>Polysemy (bank can have two meaning)</li>
  <li>Rare words</li>
</ul>


### **Q4. Effect of Embedding Dimension**
What would happen if we drastically reduce the embedding dimension (e.g., 10) or increase it (e.g., 1000)?  
Discuss the trade-offs in terms of representation power, training stability, overfitting, and downstream evaluation tasks like analogy reasoning.


<table>
    <caption>Embedding Dimension — Trade-off</caption>
    <thead>
      <tr>
        <th>Dimension</th>
        <th>Representation Power</th>
        <th>Training Stability</th>
        <th>Overfitting Risk</th>
        <th>Analogy Reasoning</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td><strong>Small (10)</strong></td>
        <td>Low</td>
        <td>Stable</td>
        <td>Low</td>
        <td>Poor</td>
      </tr>
      <tr>
        <td><strong>Large (1000)</strong></td>
        <td>High</td>
        <td>Less stable</td>
        <td>High</td>
        <td>Potentially best (if enough data)</td>
      </tr>
    </tbody>
  </table>

### **Q5. Computational Efficiency**
Word2Vec with softmax requires computing a normalization term over the entire vocabulary, which becomes computationally expensive for large $V$.  
Alternative training objectives such as **Negative Sampling** and **Hierarchical Softmax** are used to improve scalability.

- Explain why **negative sampling** makes training feasible for very large vocabularies.
- How does **hierarchical softmax**  reduce the complexity?
- Compare the computational complexity of the following methods:

  - **Full Softmax**:
  - **Negative Sampling**:
  - **Hierarchical Softmax**:

  Use Big-O notation

<b>Ans:</b><br>
For Word2Vec, predicting context word c given center word w:
$$
P(c \mid w) = \frac{exp(u_c^Tv_w)}{\sum_{w^`𝛜V}exp(u_w^Tv_w)}
$$
Denominator is sum over entire Vocabulary V that is too expensive per update.

$$
\text{Cost} = O(|V|.d)
$$

<b>Negative Sampling:</b>
Instead of computing probabilities over all words, Negative Sampling samples k randomly negative words which makes its training feasible and efficient.

$$
\text{Cost = O(k.d)} \\
\text{Where k << |V|} \\
\text{Scalable for very large vocabularies}
$$

<b>Hierarchical Softmax:</b><br> Represent vocabulary as a binary tree and each word is a leaf node. We are predicting the word by traversing path from root to leaf. At each node predict left/right child a binary decision.
$$
\text{Path Length ≈ log|V|} \\
\text{Each with cost O(d)} \\
\text{Complexity per update: O(d.log|V|)}
$$



### **Q6. t-SNE and UMAP Visualization Analysis**
After training your Word2Vec models, you plotted the embeddings using t-SNE and UMAP.  
- What insights can you derive from the spatial clustering of words?
- How do t-SNE and UMAP differ in capturing local vs global structures?
- Which plot gave more interpretable clusters and why?
- Can you identify any meaningful patterns (e.g., syntactic/semantic groupings)?

<b>Ans:</b><br>
Insights from spatial clustering of Word Embedding
<ul>
  <li>Semantic similarity: Words with similar meaning cluster together.</li>
  <li>Syntactic similarity Words of similar grammer cluster together.</li>
  <li>Topical grouping: Domain specific word form local cluster.</li>
  <li>Rare word: Can find rare word.</li>
</ul>

<b>t-SNE</b> preserver the local structure but it can be distorted cluster may appear far apart even if related. It is good to for seeing tight cluster but slower on large dataset.

<b>UMAP</b> preserves both local and some global structure. Distance between clusters are more meaningful. It is faster and scalable good for cluster and overall relationships.

UMAP plot gave more interpretable cluster because it shows cluster relationships and hierarchy.Shows the global relationship

Meaningfull Patterns:
Able to identify Part-of speech verbs, nouns, adjective form different clusters.Word with similar semantic domain form cluster.
Find rare and frequent words.

# **Task 2 : Naive Bayes**


## **1. Concepts**  
Naive Bayes is a **probabilistic machine learning algorithm** based on **Bayes’ theorem** with the simplifying assumption that all features are conditionally independent given the class. Despite its simplicity, it is widely used in domains such as spam detection, sentiment analysis, and text classification due to its efficiency and effectiveness on high-dimensional data.  

---

## **1.1 Motivation**  

- Many classification tasks involve **large feature spaces** (e.g., words in documents).  
- Complex models struggle with high dimensionality and require significant computation.  
- Naive Bayes offers a **lightweight and interpretable** solution that is both **fast to train and test**.  
- Works surprisingly well even when the independence assumption does not hold perfectly.  

**Example:** Predict whether an email is *spam* or *ham* using word occurrences.  

---

## **1.2 Core Ideas**  

## Naive Bayes Theory

- **Bayes’ Theorem**

$$
P(C \mid X) = \frac{P(X \mid C) \, P(C)}{P(X)}
$$

Where:  
- \($C$\): Class label  
- \($X$\): Feature vector (e.g., tokens in a document)  

---

### **1. Naive Independence Assumption**

$$
P(X \mid C) = \prod_{i=1}^n P(x_i \mid C)
$$


- \($X = (x_1, x_2, \dots, x_n)$\): the feature vector (all attributes/words in a document).  
- \($C$\): the class label (e.g., spam/ham, positive/negative, topic).  
- \($x_i$\): the \($i$\)-th feature of \($X$\) (e.g., presence/absence or count of word \($i$\)).  
- \($n$\): the number of features (e.g., vocabulary size).  
- \($P(X \mid C)$\): probability of observing the full feature vector \($X$\) given class \($C$\).  
- \($P(x_i \mid C)$\): probability of observing the $i$-th feature given class \($C$\).  

👉 The **naive** assumption is that features are conditionally independent given the class, so we can multiply their probabilities.

---

### **2. Classification Rule**

$$
\hat{C} = \arg\max_C \; P(C) \prod_{i=1}^n P(x_i \mid C)
$$

- \($\hat{C}$\): predicted class for the instance (the classifier’s output).  
- \($\arg\max_C$\): choose the class \($C$\) that maximizes the expression.  
- \($P(C)$\): prior probability of class \($C$\) (from class frequencies).  
- \($\prod_{i=1}^n P(x_i \mid C)$\): likelihood of observing \($X$\) under class \($C$\).  

👉 The classifier picks the class that makes the features most probable under Bayes’ rule.


- **Intuition**  
Naive Bayes simplifies classification by assuming features are conditionally independent given the class. Instead of estimating a complex joint probability distribution, it multiplies individual feature likelihoods, making it efficient and effective for high-dimensional data like text.


---

## **1.4 Implementation Details**  

## Naive Bayes Steps

1. **Training Phase**  
   - Compute **prior probabilities** $P(C)$ from class frequencies.  
   - Estimate **conditional probabilities** $P(x_i \mid C)$ from feature counts.  

---

2. **Smoothing**  
- Use **Laplace smoothing** to avoid zero probabilities:  

   $$
   P(x_i \mid C) = \frac{\text{count}(x_i, C) + 1}{\sum_j \text{count}(x_j, C) + |V|}
   $$  
where,
-  $|V|$ = vocabulary size.  
- \($x_i$\): the \($i$\)-th feature of \($X$\) (e.g., presence/absence or count of word \($i$\)).  

---

3. **Prediction Phase**  
- Compute posterior probabilities for each class.  
- Use logarithms to prevent underflow in large feature spaces.  
- Select the class with maximum posterior probability.  


## **1.5 Summary**  

- Naive Bayes is a **probabilistic classifier** grounded in Bayes’ theorem.  
- The “naive” assumption simplifies computation, making it **fast and scalable**.  
- Variants handle different data types: multinomial, Bernoulli, and Gaussian.  
- Despite its simplicity, it is one of the most **robust baseline algorithms** in machine learning, especially in **text classification**.  


## **2. Task Explanation [Implementation - $200$ marks]**

**Goal**:x

- Implement the **Naive Bayes classifier** from scratch in Python (no external ML libraries allowed; only use Python standard libraries, numpy and libraries for tokenization like CountVectorizer, TfidfVectorizer from Scikit learn or any other tokenizer as you deem fit).

- Use the below code to download the dataset for training and testing.
```python
# Expectation-Maximization
naive_bayes_train = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-naive-bayes", split="train")
naive_bayes_test = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-naive-bayes", split="test")
```

- Train your Naive Bayes model on the tokenized dataset and use it to predict class labels for the documents.    


- Save the predicted labels in a file named `nb_predictions.csv`, ensuring one label per line for each input sample. As Naive Bayes is deterministic, your results should exactly match the output from the correct implementation of naive bayes.


**Implementation Details**:

1. **Preprocessing**  
   - Perform normalization (lowercasing, punctuation removal if required).  
   - Tokenize the documents into words (you may use simple whitespace or regex-based tokenization).  

2. **Training Phase**  
   - Compute **prior probabilities** of each class.  
   - Compute **likelihood probabilities** for each word given each class using **Laplace smoothing**:  

   $$
   P(w \mid C) = \frac{\text{count}(w, C) + 1}{\sum_j \text{count}(w_j, C) + |V|}
   $$  

   where \( |V| \) is the vocabulary size.  

3. **Prediction Phase**  
   - For each test document, compute the posterior probability:  

   $$
   P(C \mid d) \propto P(C) \prod_{w \in d} P(w \mid C)
   $$  

   - Assign the label corresponding to the maximum posterior.  

4. **Evaluation**  
   - Compute **Accuracy, Precision, Recall, and F1-score** on the test set (with ground-truth labels).  
   - Save results in `nb_results.txt`.  
---
**Note**: This assignment is a way to explore various trajectories for a given problem. Clarifying every single minute detail about the implementation like hyperparameters, tolerance limit for early stopping etc. will not be entertained on Discord. You can always explore multiple paths and select the most suitable solution for the assignment. You can make assumptions about the implementation details and document it in the code. It will be highly rewarded.

---

**Deliverables**:  
- A file named `nb_predictions.csv` containing predicted labels.  
- A file named `nb_results.txt` containing evaluation metrics.  

---

**Operating Constraints**:
- DO NOT import any library except Python’s standard DO NOT import any library except Python standard library, numpy and for tokenization.
- DO NOT use any ready-made Naive Bayes implementation (e.g., from scikit-learn).  

---
---

**Proceed with clear, readable, and well-commented code!**


In [ ]:
import nltk
import os
from datasets import load_dataset
hf_token = ""

naive_bayes_train = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-naive-bayes",token=hf_token, split="train")
naive_bayes_test = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-naive-bayes", token=hf_token, split="test")

Assignment-3/naive_bayes/train_nb.parque(…):   0%|          | 0.00/190M [00:00<?, ?B/s]

Assignment-3/naive_bayes/test_nb_with_la(…):   0%|          | 0.00/47.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
df_naive_bayes_train = naive_bayes_train.to_pandas()
df_naive_bayes_test = naive_bayes_test.to_pandas()

In [ ]:
print(df_naive_bayes_test)

                                                   text       category
0     Vanuatu\nRepublic of Vanuatu | |\n---|---|\nMo...           food
1     Sports in Denver\nThis is a list of sports in ...         sports
2     Abstract algebra\nIn mathematics, more specifi...      education
3     Gastroesophageal reflux disease\nGastroesophag...           food
4     List of animals displaying homosexual behavior...         animal
...                                                 ...            ...
1995  Overview of the events of 2001 in video games\...  entertainment
1996  History of film technology\nThe history of fil...        history
1997  Animal sacrifice\nAnimal sacrifice is the ritu...         animal
1998  British Empire in World War II\nWhen the Unite...        history
1999  Escherichia coli\nEscherichia coli | |\n---|--...           food

[2000 rows x 2 columns]


In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
from collections import defaultdict, Counter
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import math
import csv

In [ ]:
import gc
gc.collect()

0

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')

STOPWORDS = set(stopwords.words('english'))


def preprocess(text):
    if isinstance(text, list):
        text = " ".join(text)
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in STOPWORDS]
    return tokens


# demo_token = preprocess(naive_bayes_train[0]["text"])
# print(demo_token)

def train_naive_bayes(texts, labels):
    class_word_counts = defaultdict(Counter)
    class_doc_counts = Counter()
    vocabulary = set()

    # Count words and documents per class
    for text, label in tqdm(zip(texts, labels), total=len(texts), desc="Counting words"):
        tokens = preprocess(text)
        class_doc_counts[label] += 1
        class_word_counts[label].update(tokens)
        vocabulary.update(tokens)

    total_docs = sum(class_doc_counts.values())
    vocab_size = len(vocabulary)

    # Compute priors
    priors = {cls: class_doc_counts[cls] / total_docs for cls in class_doc_counts}

    # Compute likelihoods with Laplace smoothing
    likelihoods = {}
    for cls in class_word_counts:
        total_words = sum(class_word_counts[cls].values())
        likelihoods[cls] = {
            word: (class_word_counts[cls][word] + 1) / (total_words + vocab_size)
            for word in vocabulary
        }

    return priors, likelihoods, vocabulary




texts = df_naive_bayes_train["text"]
labels = df_naive_bayes_train["category"]

processed_texts = [preprocess(text) for text in tqdm(texts, desc="Preprocessing")]
train_texts = processed_texts
train_labels = labels

# train_texts, test_texts, train_labels, test_labels = train_test_split(processed_texts, labels, test_size=0.2, random_state=42)


priors, likelihoods, vocabulary = train_naive_bayes(train_texts, train_labels)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Counting words: 100%|██████████| 8000/8000 [03:12<00:00, 41.46it/s]


In [ ]:
test_texts = df_naive_bayes_test["text"]
test_labels = df_naive_bayes_test["category"]

def predict(tokens, priors, likelihoods, vocabulary):
    scores = {}
    for cls in priors:
        log_prob = math.log(priors[cls])
        for word in tokens:
            if word in vocabulary:
                log_prob += math.log(likelihoods[cls].get(word, 1 / len(vocabulary)))
        scores[cls] = log_prob
    return max(scores, key=scores.get)


def evaluate(test_texts, test_labels, priors, likelihoods, vocabulary):
    print("\n🔹 Evaluating on test data...")
    preds = []
    for text in tqdm(test_texts, desc="Predicting"):
        tokens = preprocess(text)
        preds.append(predict(tokens, priors, likelihoods, vocabulary))

    acc = accuracy_score(test_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='weighted')

    with open("nb_predictions.csv", "w", newline="", encoding="utf-8") as f:
      writer = csv.writer(f)
      for label in preds:
          writer.writerow([label])

    return acc, precision, recall, f1, preds

acc, precision, recall, f1, preds = evaluate(test_texts, test_labels, priors, likelihoods, vocabulary)

with open("nb_results.txt", "w") as f:
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")
    f.write(f"F1-score: {f1:.4f}\n")

print("\n Results:")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")



## **3. Analysis (Critical Thinking & Exploration) [$5 \times 20 = 100$ marks]**

**You must answer the following five questions after implementing Naive Bayes' Classifier (Complete these in this colab file):**



**Note**:  Answers should reflect your critical thought and, where possible, relate to your classifier model's architecture, output or performance.

Q.1 Why is Naive Bayes considered a generative model, and   how does this differ from a discriminative model?


<b>Ans:</b>
Naive Bayes models the joint probability of features <i>X</i> and class <i>Y</i>
$$
P(X, Y) = P(Y) \cdot P(X \mid Y)
$$
This means it learns:


1.   Prior Probability
2.   Likelihood

Therefore, during prediction it uses Bayes Theorem:
$$
P(Y \mid X) = \frac{P(X \mid Y) \cdot P(Y)}{P(X)}
$$

So, Naive Bayes generates the distribution of features for each classe.


Discriminative Models don't model how data is generated.
Instead they learned decision boundary between class by a decision function

$$
  f(X)\rightarrow Y
$$
Therefore, it is called a generative model — it models how the data could be generated given a class.


Q.2 What is the role of the conditional independence assumption in Naive Bayes, and why is it often called “naive”?

<b>Ans:</b>
Naives Bayes needs to computes the likelihood terms:
$$
P(X \mid Y)
$$
s.t
$$
X = (X_1, X_2, \ldots, X_n)
$$
this involve exponentially many combinations - Which is very hard to compute with limited data.

So, Naive Bayes makes a simplifying assumption:
Given the class Y, all features X<sub>i</sub> are conditionally independent of each other.

$$
P(X \mid Y) = \prod_{i=1}^{n} P(X_i \mid Y)
$$	​
So this assumptions reduces the complexity and make probability estimation feasible.

Reason why it is called "Naive":
Because in real world data, features are rarely truly independent.


Q3. Why is Laplace (add-one) smoothing important in Naive Bayes, and what would happen if we do not use it?

<b>Ans</b><br>
In Naive Bayes we compute the likelihood for each feature X<sub>i</sub> given a class Y:

$$
P(X_i \mid Y) = \frac{\text{count}(X_i \text{ in Class} Y)}{\text{total words in Class } Y}
$$
But if a word appeared in a docuement but never appeared in the training data for that class then its count become 0.
$$
P(X_i \mid Y) = 0
$$
And since Naive Bayes multiplies the probabilities of all word:
$$
P(X_i \mid Y) = P(Y).\prod_{i=1}^{n}P(X_i \mid Y)
$$
Therefore, if any term become zero than the entire product become zero and the model cannot predict the class.

So to fix this we add 1 to all counts in numerator and add V vocabulary size i.e total distinct words.

$$
P(X_i \mid Y) = \frac{\text{count}(X_i \text{ in } Y) + 1}{\text{total words in Class } Y + V}\\ \text{where V is vocabulary size}
$$

By doing this


*   No word will have zero probability
*   Probabilites are slightly adjusted but remain valid.








Q4. Let's compare topic classification done here using Naive Bayes to the topic classfication task in Assignment-1.
  - *Q4-a.* Comparing Naive Bayes to K-Nearest Neighbours, which method has more training and inference time complexity and why? (10 Marks)
  - *Q4-b.* Why does Naive Bayes perform better/worse than KNN? (10 Marks)

<b>Ans(4a):</b><br>
In <b>Naive Bayes</b> we compute priors and likelihood. So we are counting feature occurence per class. So the complexity become
$$
O(N.d)
$$
Where<br>
N = Number of training samples<br> d = No. of features per sample

Advantages is Fast training-just counting, no iterative optimization.
<b>Inference Complexity </b>
$$
P(Y \mid X)  = P(Y).\prod_{i=1}^{d}P(X_i \mid Y)
$$
Complexity per test case:
$$
O(C.d)
$$
Where C = Number of classes<br>
Advantages: Very fast no need to scan all training data again
<hr>

In <b>KNN</b> during training it just stores all training data.<br>
Complexity:
$$
O(1)\text{ just storing data}
$$
very fast to train but storage-intensive
<br>Inference Complexity:<br>
To classify a test sample, KNN must compute distance to every distance point.
$$
O(N.d)
$$
Pick k nearest neighbours.<br>
N = No of training set size.
<br> It can be slow on large data, especially when with high-dimension data


<b>Ans(4b):</b><br>
Naives Bayes works better than KNN
<ul>
<li>Works well with high dimensional data. </li>
<li>Fast Inference</li>
<li>Robust to irrelevant features.</li>
<li>Small training sets.</li>
</ul>

Naives Bayes works worse than KNN
<ul>
<li>Conditional Independence assumption is often violated.</li>
<li>Poor with small dataset.</li>
<li>Sensitive to Laplace smoothing choice</li>
</ul>


Q5. Why is Naive Bayes often said to perform well in text classification problems, despite its simplistic assumptions?

<b>Ans:</b><br>
Naives Bayes assumes that word are independent given the class
$$
P(X \mid Y) = \prod_{i=1}^{n}P(X_i \mid Y)
$$
In reality words are correlated but the relative likelihoods are usually sufficient for the classificaition problem.<br>
<br>

Text data has thousands of feature(words) most of which are zero in any docuement (sparse). But Naive Bayes  

*   Only needs word counts per class
*   Ignores zero entries automatically
*   Efficiently handles large vocabularies

Smoothing Helps:
Laplace smoothing prevents zero probablities, which allows the model to handle unseen words gracefully.






# **Task 3 : Expectation Maximization**

## **1. Concepts**

Expectation Maximization (EM) is a **probabilistic optimization algorithm** used to estimate parameters of statistical models when data has **latent (hidden) variables**. It alternates between assigning probabilities to hidden variables (E-step) and maximizing the likelihood of the parameters (M-step). EM is widely applied in clustering (e.g., Gaussian Mixture Models), missing data problems, and probabilistic inference.  

---

## 1.1 Motivation

- Real-world datasets often contain **incomplete or hidden information** (e.g., cluster assignments are unknown).  
- Direct maximization of the likelihood function becomes intractable due to hidden variables.  
- EM provides an **iterative framework** to estimate parameters efficiently by breaking the problem into two simpler steps.  

Example:  
Clustering points into multiple Gaussian distributions when class labels are unknown.  

---

## 1.2 Core Ideas

### (A) Theoretical Explanation

- **Likelihood with hidden variables**:  

$$
P(X \mid \theta) = \sum_Z P(X, Z \mid \theta)
$$  

Where:  
- \( $X$ \): Observed data  
- \( $Z$ \): Latent (hidden) variables  
- \( $\theta$ \): Model parameters  

- **E-step (Expectation)**:  
  Estimate the posterior distribution of hidden variables given current parameters:  

$$
Q(Z) = P(Z \mid X, \theta^{(t)})
$$  

- **M-step (Maximization)**:  
  Update parameters by maximizing the expected log-likelihood:  

$$
\theta^{(t+1)} = \arg\max_\theta \, \mathbb{E}_{Q(Z)}[\log P(X, Z \mid \theta)]
$$  

- **Intuition**:  
  The E-step “fills in” missing/hidden data with probabilities, while the M-step re-estimates parameters as if the hidden data were observed. This alternation improves the likelihood iteratively.  

---

### (B) Example: Gaussian Mixture Model (GMM) Clustering  

Given data points without labels:  

- **E-step**: Compute the probability that each data point belongs to each Gaussian component.  
- **M-step**: Update means, covariances, and mixture weights using these probabilities.  
- Repeat until convergence.  

This way, EM allows clustering without prior labels.  

---

## 1.3 Variants of EM  

1. **Gaussian Mixture Models (GMM-EM)**  
   - Clustering with Gaussian distributions.  

2. **Hidden Markov Models (HMM-EM, i.e., Baum-Welch Algorithm)**  
   - Sequence data with hidden states.  

3. **Soft K-Means**  
   - EM with isotropic Gaussians and uniform variance assumption.  

---

## 1.4 Implementation Details  

1. **Initialization**  
   - Start with random guesses for parameters (e.g., means and covariances in GMM).  

2. **E-step**  
   - Calculate the probability distribution of hidden variables given observed data.  

3. **M-step**  
   - Update parameter estimates by maximizing the expected log-likelihood.  

4. **Convergence**  
   - Stop when parameters stabilize or the log-likelihood improvement falls below a threshold.   

---

## 1.5 Summary  

- Expectation Maximization is an **iterative optimization algorithm** for parameter estimation in models with hidden variables.  
- Alternates between **E-step (inferring hidden variables)** and **M-step (optimizing parameters)**.  
- Widely applied in **clustering, HMMs, and incomplete-data problems**.  
- Sensitive to initialization but remains a cornerstone in unsupervised learning.  

---



## **2. Task Explanation [Implementation - $200$ marks]**

**Goal**:

- Implement the **Expectation-Maximization (EM) Algorithm** from scratch in Python (no external ML libraries allowed; only use Python standard libraries, numpy and libraries for tokenization like CountVectorizer, TfidfVectorizer from Scikit learn or any other tokenizer as you deem fit).

- Use the below code to download the dataset for training and testing.
```python
# Expectation-Maximization
em_train = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-em", split="train")
em_test = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-em", split="test")
```

- Use BOW.

- Apply the EM algorithm to cluster the tokenized dataset into a predefined number of clusters (10). Treat each document as a bag of tokens.

- Save the predicted cluster assignments to a file named `em_predictions.csv`, with one cluster number per line (corresponding to each input document).

- **Adjusted Rand Index (ARI)** will be used to evaluate your predictions with respect to the true labels of test set. ARI is a clustering evaluation metric that considers all pairs of samples in the dataset and for each  each pair, it checks whether the samples are:
  - In the same cluster in both true and predicted labels (agreement ✅)

  - In different clusters in both true and predicted labels (agreement ✅)

  - In the same cluster in one but different in the other (disagreement ❌)

  - A score of 0 represents random clustering in the scale of [-1,+1]. A well imppelemted EM algorithm should yield positive ARI scores.

---
- **Note**: This assignment is a way to explore various trajectories for a given problem. Clarifying every single minute detail about the implementation like hyperparameters, tolerance limit for early stopping etc. will not be entertained on Discord. You can always explore multiple paths and select the most suitable solution for the assignment. You can make assumptions about the implementation details and document it in the code. It will be highly rewarded.

---

- **Deliverables**:  
  - `em_predictions.csv`
---

- **Operating constraints**:  
  - DO NOT import any library except Python standard library, numpy and for tokenization.  
  - DO NOT use any ready-made EM algorithm or clustering implementation.

---
---

**Proceed with clear, readable, and well-commented code!**

In [ ]:
import numpy as np
import os
import csv
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm
np.random.seed(42)
hf_token = ""

In [ ]:
em_train = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-em", token = hf_token, split="train")


README.md:   0%|          | 0.00/6.20k [00:00<?, ?B/s]

Assignment-3/em/train_em.parquet:   0%|          | 0.00/189M [00:00<?, ?B/s]

Assignment-3/em/test_em.parquet:   0%|          | 0.00/47.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
print(em_train)

Dataset({
    features: ['text', 'category'],
    num_rows: 8000
})


In [ ]:
unique = em_train.unique("category")
print(unique)

[None]


In [ ]:
texts = em_train["text"]

In [ ]:
vectorizer = CountVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(texts).toarray()   # shape: (N, vocab_size)

N, D = X.shape
K = 10  # number of clusters


In [ ]:
import numpy as np
import csv
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer

#Tokenize bag of words

vectorizer = CountVectorizer(max_features=25)
X = vectorizer.fit_transform(texts).toarray()  # shape: (n_docs, n_features)
N, D = X.shape
K = 10  # number of clusters

In [ ]:
import numpy as np
import csv

# Initialize parameters
np.random.seed(42)
pi = np.ones(K) / K
means = X[np.random.choice(N, K, replace=False)]
covs = np.ones((K, D))

def gaussian_log_prob_diag(X, means, covs):

    X_exp = X[:, None, :]
    diff = X_exp - means

    log_det = -0.5 * np.sum(np.log(2 * np.pi * covs), axis=1)
    mahalanobis = -0.5 * np.sum(diff**2 / covs, axis=2)
    log_prob = mahalanobis + log_det

    return log_prob

def logsumexp(a, axis=1):
    a_max = np.max(a, axis=axis, keepdims=True)
    result = a_max + np.log(np.sum(np.exp(a - a_max), axis=axis, keepdims=True))
    return np.squeeze(result, axis=axis)

# EM Algorithm

max_iters = 50
prev_log_likelihood = -np.inf
log_likelihood_history = []

for iteration in range(max_iters):

    # Computing log responsibilities
    log_probs = gaussian_log_prob_diag(X, means, covs)
    weighted_log_probs = log_probs + np.log(pi + 1e-10)

    # Computing log-likelihood
    log_likelihood = np.sum(logsumexp(weighted_log_probs, axis=1))
    log_likelihood_history.append(log_likelihood)

    #computing responsibilities
    log_resp = weighted_log_probs - logsumexp(weighted_log_probs, axis=1)[:, None]
    resp = np.exp(log_resp)
    resp_sum = resp.sum(axis=1, keepdims=True)
    resp = resp / resp_sum

    Nk = resp.sum(axis=0) + 1e-10
    pi = Nk / N
    means = (resp.T @ X) / Nk[:, None]

    # update Diagonal covariance with better regularization
    diff = X[:, None, :] - means
    covs = (resp[:, :, None] * diff**2).sum(axis=0) / Nk[:, None]

    min_cov = 1e-3 * np.median(covs)
    covs = np.maximum(covs, min_cov)

    improvement = log_likelihood - prev_log_likelihood
    print(f"Iteration {iteration+1}, Log-Likelihood: {log_likelihood:.3f}, Improvement: {improvement:.6f}")

    if iteration > 0 and improvement < 1e-6:
        print(f"Converged after {iteration+1} iterations")
        break

    prev_log_likelihood = log_likelihood

print("\n=== Final Results ===")
print("Log-likelihood progression:", [f"{x:.3f}" for x in log_likelihood_history])
print("Cluster sizes:", np.bincount(np.argmax(resp, axis=1)))
print("Final covariances range:", f"[{np.min(covs):.6f}, {np.max(covs):.6f}]")

# Verify monotonic increase
increasing = all(log_likelihood_history[i] <= log_likelihood_history[i+1]
                 for i in range(len(log_likelihood_history)-1))
print(f"Log-likelihood monotonically increasing: {increasing}")



clusters = np.argmax(resp, axis=1)
with open("em_predictions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for c in clusters:
        writer.writerow([int(c)])

print("Predictions saved to em_predictions.csv.")

Iteration 1, Log-Likelihood: -285501586.816, Improvement: inf
Iteration 2, Log-Likelihood: -874496.758, Improvement: 284627090.057724
Iteration 3, Log-Likelihood: -860322.984, Improvement: 14173.774078
Iteration 4, Log-Likelihood: -856105.669, Improvement: 4217.314441
Iteration 5, Log-Likelihood: -854598.084, Improvement: 1507.584997
Iteration 6, Log-Likelihood: -853971.633, Improvement: 626.451766
Iteration 7, Log-Likelihood: -853674.215, Improvement: 297.417262
Iteration 8, Log-Likelihood: -853497.319, Improvement: 176.896644
Iteration 9, Log-Likelihood: -853382.907, Improvement: 114.411944
Iteration 10, Log-Likelihood: -853292.744, Improvement: 90.162432
Iteration 11, Log-Likelihood: -853223.386, Improvement: 69.358665
Iteration 12, Log-Likelihood: -853173.348, Improvement: 50.037857
Iteration 13, Log-Likelihood: -853131.648, Improvement: 41.699589
Iteration 14, Log-Likelihood: -853096.203, Improvement: 35.445153
Iteration 15, Log-Likelihood: -853063.060, Improvement: 33.142726
Iter

In [ ]:
em_test = load_dataset("Exploration-Lab/CS779-Fall25", "Assignment-3-em", token = hf_token, split="test")


In [ ]:
print(em_test)

Dataset({
    features: ['text', 'category'],
    num_rows: 2000
})


In [ ]:
#Inference Phase

vectorizer = CountVectorizer(max_features=25)
X = vectorizer.fit_transform(em_test["text"]).toarray()
N, D = X.shape
K = 10


def gaussian_log_prob_diag(X, means, covs):


    X_exp = X[:, None, :]
    diff = X_exp - means

    log_det = -0.5 * np.sum(np.log(2 * np.pi * covs), axis=1)
    mahalanobis = -0.5 * np.sum(diff**2 / covs, axis=2)
    log_prob = mahalanobis + log_det

    return log_prob

def predict_clusters(X, trained_means, trained_covs, trained_pi):

    # Compute log probabilities
    log_probs = gaussian_log_prob_diag(X, trained_means, trained_covs)  # (N, K)

    # Weight by cluster probabilities
    weighted_log_probs = log_probs + np.log(trained_pi + 1e-10)  # (N, K)

    # Assign to most probable cluster
    clusters = np.argmax(weighted_log_probs, axis=1)

    return clusters

def predict_probabilities(X, trained_means, trained_covs, trained_pi):

    #Get soft cluster assignments for new data
    def logsumexp(a, axis=1):
        a_max = np.max(a, axis=axis, keepdims=True)
        result = a_max + np.log(np.sum(np.exp(a - a_max), axis=axis, keepdims=True))
        return np.squeeze(result, axis=axis)

    # Compute log probabilities
    log_probs = gaussian_log_prob_diag(X, trained_means, trained_covs)
    weighted_log_probs = log_probs + np.log(trained_pi + 1e-10)

    # Convert to probabilities using softmax
    log_resp = weighted_log_probs - logsumexp(weighted_log_probs, axis=1)[:, None]
    probabilities = np.exp(log_resp)

    return probabilities


trained_means = means
trained_covs = covs
trained_pi = pi



# Make predictions
clusters = predict_clusters(X, trained_means, trained_covs, trained_pi)

probabilities = predict_probabilities(X, trained_means, trained_covs, trained_pi)

with open("em_predictions_test.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for c in clusters:
        writer.writerow([int(c)])

print("Predictions saved")

print(f"\n=== Inference Results ===")
print(f"Data shape: {X.shape}")
print(f"Number of clusters: {len(trained_pi)}")
print(f"Cluster distribution: {np.bincount(clusters)}")
print(f"Cluster weights: {trained_pi}")

Predictions saved

=== Inference Results ===
Data shape: (2000, 25)
Number of clusters: 10
Cluster distribution: [226 133 112 277 321 227 204 167 237  96]
Cluster weights: [0.11213911 0.0584733  0.0580955  0.14113444 0.15586801 0.11611573
 0.09102004 0.0899868  0.12887415 0.04829292]



## **3. Analysis (Critical Thinking & Exploration) [$5 \times 20 = 100$ marks]**

**You must answer the following five questions after implementing Expectation Maximization algorithm (Complete these in this colab file):**



**Note**:  Answers should reflect your critical thought and, where possible, relate to your algorithm and its performance.

Q.1 Why is Expectation Maximization considered an iterative optimization algorithm, and how does it differ from direct likelihood maximization?

<b>Ans:</b><br>
Expectation-Maximization (EM) is used when maximum likelihood estimation (MLE) is difficult to compute directly due to latent (hidden) variables. It is considered as an iterative optimization algorithm because it maximizes the likelihood iteratively by alternating between:<br>
1) E-step (Expectation step): Compute expected value of the complete-data log-likelihood.<br>
2) M-step (Maximization step):Maximize this expected log-likelihood to update parameters.

Repeat until convergence.<br>
Each iteration is guaranteed to not decrease the likelihood.

Q.2 What is the role of the E-step and M-step in EM, and why are they repeated until convergence?  

<b>Ans:</b><br>
Role of the E-step (Expectation Step)<br>
Goal: Estimate the missing or latent variable using the current parameter estimates.<br>
It computes the exptected value of the complete-data log-likelihood with respect to the distribution of Z.<br>

Role of the M-step(Maximization Step)<br>
Goal: Update the model parameter θ to maximize the expected complete data-log likelihood computed in E-step<br>

Repeat Unitl Convergence
Each E-step estimates latent variables given current parameter and each M-step updates the parameter using those estimates.<br>

Convergence criterion: Repeat until the change in likelihood is below a threshold.<br>

We iterate until the model stablize at a local maximum of likelihood.


Q.3 Why is EM sensitive to initialization, and how can this problem be mitigated in practice?  

<ul>

  <li>Local Maxima Problem: EM maximizes the likelihood iteratively but is not guaranteed to find the global maximum. It converges to a local maximum of the likelihood surface, which depends heavily on the starting parameters.</li>

  <li>The E-step computes expected values of hidden variables based on current parameters.If the initial parameters are poor, these “soft assignments” can be misleading, leading M-step into a bad solution.</li>

  <li>For models like Gaussian Mixture Models (GMMs) with many clusters, the likelihood surface may have many peaks.</li>
</ul>

Solution:
<ul>
  <li>Multiple random restarts: Run EM multiple times with different initial parameter and choose the solution with highest final likelihood.</li>

  <li>Can use K-means to initialize cluster centers.</li>

  <li>We can train EM on small dataset first to get reasonalbe initial parameter.</li>
</ul>

Q.4 What are some advantages and disadvantages of EM compared to other clustering/optimization algorithms (e.g., K-Means, gradient-based methods)?  

<b>Ans:</b><br>
<b>Advantages of EM</b><br>
<ul>
  <li>Handles latent variable naturally EM can model hidden structure in data.</li>
  <li>Probablistic framework provides soft assignments rather than hard labels.</li>
  <li>It can be applied to any model with latent variable.
</ul>

<b>Disadvantages</b><br>
<ul>
  <li>Can converge to local maximum </li>
  <li>Requires computing expectation over large latent spaces makes it slower than k-means</li>
  <li>Unlike convex gradient-based optimization, EM does not guarantee global optimum.</li>
</ul>

Q.5 Compare the results obtained in EM with results obtained from Naive bayes. Also delve upon how can we predict labels of a test dataset without labels after EM and why it can work?

<b>Ans:</b><br>
<b>EM Algorithm: </b><br>
<ul>
  <li>Expectation-Maximization (EM) is an unsupervised iterative optimization algorithm used to learn latent structures in data.</li>
  <li>It output the soft cluster assignments</li>
  <li>Latent variable model often assumes Gaussian or mixture distributions</li>
  <li>Good at discovering structure; may not perfectly align with true labels</li>
  <li>High (local maxima possible)</li>
</ul>

<b> Naive Bayes: </b><br>
<ul>
  <li>Naive Bayes is a supervised probabilistic classifier that predicts labels based on labeled training data.</li>
  <li>Direct label prediction. </li>
  <li>Conditional independence of features given class</li>
  <li>Optimized for label prediction; accurate with labeled data</li>
  <li>Low (deterministic given labeled data)</li>
</ul>
